---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("="*70)
print("ADAMSENSE DATA EXPLORATION")
print("Wrist-Worn Activity Recognition")
print("="*70)
print("\n✓ Libraries loaded successfully")

---
## Step 2: Configuration - UPDATE THIS PATH!

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THIS PATH!
# ============================================================================

DATASET_PATH = '../data/raw/AdamSense.csv'  

# Output directory for figures
OUTPUT_DIR = './exploration_outputs/'

# ============================================================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Dataset: {DATASET_PATH}")
print(f"  Output:  {OUTPUT_DIR}")
print(f"\n✓ Configuration complete")

---
## Step 3: Load AdamSense Dataset

In [ ]:
print("="*70)
print("LOADING ADAMSENSE DATASET")
print("="*70)

if not Path(DATASET_PATH).exists():
    print("\nPlease update DATASET_PATH in Step 2.")
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

print(f"\nLoading: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)

print(f"\n✓ Dataset loaded successfully!")
print(f"\nBasic Info:")
print(f"  Total samples: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nAll columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")

---
## Step 4: Identify Wrist Sensor Columns

In [ ]:
print("="*70)
print("IDENTIFYING WRIST SENSOR COLUMNS")
print("="*70)

# AdamSense has two devices: pocket (_p) and wrist (_w)
# We only need wrist data for our microcontroller

# Check for wrist columns
wrist_accel = ['Ax_w', 'Ay_w', 'Az_w']
wrist_gyro = ['Gx_w', 'Gy_w', 'Gz_w']
wrist_sensors = wrist_accel + wrist_gyro

# Verify all columns exist
missing_cols = [col for col in wrist_sensors if col not in df.columns]

if missing_cols:
    print(f"\nERROR: Missing wrist sensor columns: {missing_cols}")
    print(f"\nAvailable columns: {list(df.columns)}")
    raise ValueError("Required wrist sensor columns not found")

print("\nAll wrist sensor columns found!")
print("\nWrist Sensors (for microcontroller):")
print(f"  Accelerometer: {wrist_accel}")
print(f"  Gyroscope:     {wrist_gyro}")
print(f"  Total: 6 channels")

# Check for magnetometer (we won't use it, but good to know)
wrist_mag = ['Mx_w', 'My_w', 'Mz_w']
has_magnetometer = all(col in df.columns for col in wrist_mag)

if has_magnetometer:
    print(f"\n Magnetometer data present: {wrist_mag}")
    print("   → Will NOT be used (microcontroller doesn't have magnetometer)")

# Check for timestamp
if 'Time_w' in df.columns:
    timestamp_col = 'Time_w'
    print(f"\n✓ Wrist timestamp column found: {timestamp_col}")
else:
    print(f"\n Warning: 'Time_w' column not found")
    print(f"   Available time columns: {[col for col in df.columns if 'time' in col.lower()]}")
    timestamp_col = None

---
## Step 5: Basic Dataset Information

In [ ]:
print("="*70)
print("DATASET OVERVIEW")
print("="*70)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nData types:")
display(df.dtypes)

In [ ]:
# Statistical summary for wrist sensors only
print("\nWrist Sensor Statistics:")
display(df[wrist_sensors].describe())

---
## Step 6: Activity Distribution Analysis

In [ ]:
print("="*70)
print("ACTIVITY DISTRIBUTION")
print("="*70)

activity_counts = df['Activity'].value_counts()
total = len(df)

print(f"\nTotal unique activities: {len(activity_counts)}")
print(f"\nActivity breakdown:")
print("-"*70)
print(f"{'Activity':<30} {'Count':>12} {'Percentage':>12} {'Sufficient?':>12}")
print("-"*70)

for activity, count in activity_counts.items():
    pct = count / total * 100
    sufficient = '✓' if count >= 50000 else '⚠️'
    bar = '█' * min(int(pct / 2), 50)
    print(f"{activity:<30} {count:>12,} {pct:>11.2f}% {sufficient:>12}")

print("-"*70)
print(f"{'TOTAL':<30} {total:>12,} {100.00:>11.2f}%")
print("\n✓ = Sufficient samples (≥50k) for training")
print("⚠️  = Low samples (<50k) - consider excluding")

In [ ]:
# Visualize activity distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
colors = sns.color_palette("husl", len(activity_counts))
activity_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('AdamSense Activity Distribution', fontsize=16, weight='bold', pad=20)
axes[0].set_xlabel('Activity', fontsize=12, weight='bold')
axes[0].set_ylabel('Sample Count', fontsize=12, weight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=10)
axes[0].grid(True, alpha=0.3, linestyle='--')

# Add 50k threshold line
axes[0].axhline(y=50000, color='red', linestyle='--', linewidth=2, label='50k threshold')
axes[0].legend()

# Add value labels on bars
for i, (activity, count) in enumerate(activity_counts.items()):
    axes[0].text(i, count + (total * 0.01), f'{count:,}', 
                ha='center', va='bottom', fontsize=8, weight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    activity_counts.values, 
    labels=activity_counts.index, 
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    explode=[0.05] * len(activity_counts),
    shadow=True
)
axes[1].set_title('Activity Distribution - Percentage', fontsize=16, weight='bold', pad=20)

# Enhance text
for text in texts:
    text.set_fontsize(9)
    text.set_weight('bold')
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(9)
    autotext.set_weight('bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}activity_distribution.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {OUTPUT_DIR}activity_distribution.png")
plt.show()

# Recommendations
print("\n" + "="*70)
print("RECOMMENDATION FOR ACTIVITY SELECTION")
print("="*70)

sufficient_activities = activity_counts[activity_counts >= 50000]
print(f"\nActivities with ≥50k samples: {len(sufficient_activities)}")
if len(sufficient_activities) >= 5:
    print("\n✓ Good! You have enough activities for training.")
    print("\nSuggested activities to select:")
    for i, (activity, count) in enumerate(sufficient_activities.head(10).items(), 1):
        print(f"  {i}. {activity:<30} ({count:,} samples)")
else:
    print(f"\n⚠️  Only {len(sufficient_activities)} activities have ≥50k samples.")
    print("   Consider lowering threshold or including more activities.")

---
## Step 7: User Distribution Analysis

In [ ]:
print("="*70)
print("USER DISTRIBUTION")
print("="*70)

user_counts = df['User'].value_counts().sort_index()
total_users = len(user_counts)

print(f"\nTotal users: {total_users}")
print(f"\nSamples per user statistics:")
print(f"  Minimum:  {user_counts.min():>8,} samples")
print(f"  Maximum:  {user_counts.max():>8,} samples")
print(f"  Mean:     {user_counts.mean():>8,.0f} samples")
print(f"  Median:   {user_counts.median():>8,.0f} samples")
print(f"  Std Dev:  {user_counts.std():>8,.0f} samples")

print(f"\nFirst 15 users:")
print("-"*50)
for i, (user, count) in enumerate(user_counts.head(15).items()):
    pct = count / len(df) * 100
    bar = '▓' * int(pct * 2)
    print(f"  User {user:>3d}: {count:>8,} samples ({pct:>5.2f}%) {bar}")

In [ ]:
# Visualize user distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram
axes[0, 0].hist(user_counts.values, bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(user_counts.mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {user_counts.mean():.0f}')
axes[0, 0].axvline(user_counts.median(), color='blue', linestyle='--', linewidth=2, 
                   label=f'Median: {user_counts.median():.0f}')
axes[0, 0].set_title('Distribution of Samples per User', fontsize=14, weight='bold')
axes[0, 0].set_xlabel('Number of Samples')
axes[0, 0].set_ylabel('Number of Users')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Top 20 users bar plot
user_counts.head(20).plot(kind='bar', ax=axes[0, 1], color='steelblue', edgecolor='black')
axes[0, 1].set_title('Top 20 Users by Sample Count', fontsize=14, weight='bold')
axes[0, 1].set_xlabel('User ID')
axes[0, 1].set_ylabel('Sample Count')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Activity distribution per user (first 10)
activity_user_crosstab = pd.crosstab(df['User'], df['Activity'])
activity_user_crosstab.head(10).plot(
    kind='bar', stacked=True, ax=axes[1, 0], 
    colormap='tab20', edgecolor='black', linewidth=0.5
)
axes[1, 0].set_title('Activity Distribution per User (First 10)', fontsize=14, weight='bold')
axes[1, 0].set_xlabel('User ID')
axes[1, 0].set_ylabel('Sample Count')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].legend(title='Activity', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Box plot
axes[1, 1].boxplot([user_counts.values], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightgreen', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))
axes[1, 1].set_title('Box Plot - Samples per User', fontsize=14, weight='bold')
axes[1, 1].set_ylabel('Sample Count')
axes[1, 1].set_xticklabels(['All Users'])
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}user_distribution.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {OUTPUT_DIR}user_distribution.png")
plt.show()

---
## Step 8: Wrist Sensor Statistics

In [ ]:
print("="*70)
print("WRIST SENSOR DATA STATISTICS")
print("="*70)

print("\nSensor channels: 6 (Accelerometer + Gyroscope)")
print("Target hardware: Arduino Nano 33 BLE Sense (no magnetometer)")

print("\nDetailed Statistics:")
print("-"*85)
print(f"{'Sensor':<10} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12} {'Range':>12}")
print("-"*85)

for sensor in wrist_sensors:
    mean = df[sensor].mean()
    std = df[sensor].std()
    min_val = df[sensor].min()
    max_val = df[sensor].max()
    range_val = max_val - min_val
    
    print(f"{sensor:<10} {mean:>12.4f} {std:>12.4f} {min_val:>12.4f} {max_val:>12.4f} {range_val:>12.4f}")

print("-"*85)

# Data quality checks
print("\nData Quality Checks:")

# Missing values
missing = df[wrist_sensors].isnull().sum()
if missing.sum() == 0:
    print("  ✓ No missing values detected!")
else:
    print("  ⚠️  Missing values detected:")
    for sensor, count in missing.items():
        if count > 0:
            pct = count / len(df) * 100
            print(f"    {sensor}: {count:,} ({pct:.2f}%)")

# Infinite values
infinite = df[wrist_sensors].apply(lambda x: np.isinf(x).sum())
if infinite.sum() == 0:
    print("  ✓ No infinite values detected!")
else:
    print("  ⚠️  Infinite values detected:")
    for sensor, count in infinite.items():
        if count > 0:
            print(f"    {sensor}: {count:,}")

# Zero variance check
print("\n  Sensor variance check:")
for sensor in wrist_sensors:
    var = df[sensor].var()
    if var < 0.001:
        print(f"    ⚠️  {sensor}: Very low variance ({var:.6f})")
    else:
        print(f"    ✓ {sensor}: Good variance ({var:.4f})")

In [ ]:
# Visualize sensor distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple', 'gold', 'crimson']
sensor_names = ['Accel X', 'Accel Y', 'Accel Z', 'Gyro X', 'Gyro Y', 'Gyro Z']

for i, (sensor, name, color) in enumerate(zip(wrist_sensors, sensor_names, colors)):
    # Histogram
    axes[i].hist(df[sensor].dropna(), bins=60, color=color, edgecolor='black', alpha=0.7, linewidth=0.5)
    
    # Add mean and median lines
    mean_val = df[sensor].mean()
    median_val = df[sensor].median()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    axes[i].axvline(median_val, color='blue', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
    
    axes[i].set_title(f'{name} ({sensor})', fontsize=12, weight='bold')
    axes[i].set_xlabel('Value', fontsize=10)
    axes[i].set_ylabel('Frequency', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3, linestyle='--')

plt.suptitle('Wrist Sensor Distributions - AdamSense', fontsize=16, weight='bold', y=1.00)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}sensor_distributions.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {OUTPUT_DIR}sensor_distributions.png")
plt.show()

---
## Step 9: SAMPLING RATE CALCULATION - CRITICAL!

In [ ]:
print("="*70)
print("SAMPLING RATE - FROM DOCUMENTATION")
print("="*70)

# According to AdamSense documentation (page 5, section 8, point 5):
# "The sample rate for smart phone and IMU both is kept 50Hz"

final_sampling_rate = 50 

print("\n✓ OFFICIAL SAMPLING RATE: 50 Hz")

print("\n" + "="*70)
print("WINDOW SIZE CALCULATIONS")
print("="*70)

print(f"\nAt 50 Hz sampling rate:")
print("-"*70)

durations = [2.0, 2.5, 3.0, 4.0]
for duration in durations:
    samples = int(50 * duration)
    power_of_2 = 2 ** round(np.log2(samples))
    print(f"  {duration}s = {samples:>3d} samples ≈ {power_of_2:>3d} samples (power of 2)")

recommended_window = 128
print(f"\n RECOMMENDED: {recommended_window} samples")
print(f"   Duration: {recommended_window/50:.2f} seconds")
print(f"   Matches your StressSense setup!")

print("\n" + "="*70)
print("DATA COLLECTION DETAILS")
print("="*70)

print("\nFrom documentation:")
print(f"  • Sampling rate: 50 Hz")
print(f"  • Duration per activity: ~2 minutes")
print(f"  • Total subjects: 10")
print(f"  • Total activities: 11")
print(f"  • Total instances: 709,582")
print(f"  • Sensors: SparkFun 9DoF Razor IMU M0")

print("\n" + "="*70)

---
## Step 10: Visualize Activity Patterns

In [ ]:
print("="*70)
print("ACTIVITY SENSOR PATTERNS")
print("="*70)

# Select up to 6 activities for visualization
unique_activities = df['Activity'].unique()[:6]
n_activities = len(unique_activities)

print(f"\nVisualizing {n_activities} activities...")

fig, axes = plt.subplots(n_activities, 1, figsize=(16, n_activities * 2.5))
if n_activities == 1:
    axes = [axes]

sample_size = 200  # samples to plot

for i, activity in enumerate(unique_activities):
    activity_data = df[df['Activity'] == activity].head(sample_size)
    
    # Plot all 6 channels
    axes[i].plot(activity_data['Ax_w'].values, label='Accel X', alpha=0.8, linewidth=1.5, color='#1f77b4')
    axes[i].plot(activity_data['Ay_w'].values, label='Accel Y', alpha=0.8, linewidth=1.5, color='#ff7f0e')
    axes[i].plot(activity_data['Az_w'].values, label='Accel Z', alpha=0.8, linewidth=1.5, color='#2ca02c')
    axes[i].plot(activity_data['Gx_w'].values, label='Gyro X', alpha=0.6, linewidth=1.2, linestyle='--', color='#d62728')
    axes[i].plot(activity_data['Gy_w'].values, label='Gyro Y', alpha=0.6, linewidth=1.2, linestyle='--', color='#9467bd')
    axes[i].plot(activity_data['Gz_w'].values, label='Gyro Z', alpha=0.6, linewidth=1.2, linestyle='--', color='#8c564b')
    
    axes[i].set_ylabel(activity, fontsize=11, weight='bold', rotation=0, ha='right', va='center')
    axes[i].legend(loc='upper right', fontsize=9, ncol=6)
    axes[i].grid(True, alpha=0.3, linestyle='--')
    axes[i].set_xlim(0, sample_size)

axes[-1].set_xlabel('Sample Index', fontsize=12, weight='bold')
plt.suptitle('Wrist Sensor Patterns for Different Activities', fontsize=16, weight='bold', y=0.995)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}activity_patterns.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {OUTPUT_DIR}activity_patterns.png")
plt.show()

---
## Step 11: Correlation Analysis

In [ ]:
print("="*70)
print("SENSOR CORRELATION ANALYSIS")
print("="*70)

# Calculate correlation matrix
corr_matrix = df[wrist_sensors].corr()

print("\nCorrelation matrix:")
display(corr_matrix)

# Visualize
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            xticklabels=['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz'],
            yticklabels=['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz'])
plt.title('Wrist Sensor Correlation Matrix - AdamSense', fontsize=14, weight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}correlation_matrix.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {OUTPUT_DIR}correlation_matrix.png")
plt.show()

# Find highly correlated pairs
print("\nHighly correlated sensor pairs (|r| > 0.7):")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

if high_corr:
    for sensor1, sensor2, corr_val in high_corr:
        print(f"  {sensor1} <-> {sensor2}: {corr_val:.3f}")
else:
    print("  None found (good - sensors are independent)")

---
## Step 12: FINAL SUMMARY REPORT

In [ ]:
print("="*70)
print("ADAMSENSE EXPLORATION - FINAL SUMMARY")
print("="*70)

print(f"\n  Total samples:      {len(df):,}")
print(f"  Total activities:   {len(activity_counts)}")
print(f"  Total users:        {len(user_counts)}")
print(f"  Sensor channels:    {len(wrist_sensors)} (Accel + Gyro)")
print(f"  Sampling rate:      {final_sampling_rate:.0f} Hz")
print(f"  Recommended window: {recommended_window} samples ({recommended_window/final_sampling_rate:.2f}s)")
print(f"  Missing values:     {df[wrist_sensors].isnull().sum().sum()}")

print(f"\n  Most common:        {activity_counts.index[0]} ({activity_counts.iloc[0]:,})")
print(f"  Least common:       {activity_counts.index[-1]} ({activity_counts.iloc[-1]:,})")
print(f"  Imbalance ratio:    {activity_counts.iloc[0] / activity_counts.iloc[-1]:.2f}x")

print(f"\nGenerated files in {OUTPUT_DIR}:")
for f in ['activity_distribution.png', 'user_distribution.png', 'sensor_distributions.png',
          'activity_patterns.png', 'correlation_matrix.png']:
    print(f"  - {f}")

print("\n" + "="*70)
print("EXPLORATION COMPLETE")
print("="*70)
